# Imports

In [1]:
import pandas as pd
import numpy as np

from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

import os

import sqlite3
import faiss

/home/mp/Desktop/grocery_swiper/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load data

In [2]:
CSV_PATH = "../data/csv/processed/"
files    = os.listdir(CSV_PATH)
csvs     = sorted([file.split('_')[1].split('.')[0] for file in files])

historic_csvs = csvs[:-1]
train_df      = pd.concat([pd.read_csv(f"{CSV_PATH}products_{csv}.csv") for csv in historic_csvs], ignore_index=True)

latest_csv = csvs[-1]
test_df    = pd.read_csv(f"{CSV_PATH}products_{latest_csv}.csv")

In [3]:
df = pd.read_csv("../data/csv/mail_groceries.csv")

conn = sqlite3.connect("../data/sql/swipes.db")
labels = pd.read_sql_query("SELECT data_id, is_liked, is_superliked, is_passed FROM swipes", conn)
conn.close()

In [4]:
labels

,data_id,is_liked,is_superliked,is_passed
0,10876682,0,0,1
1,10808108,1,0,0
2,10863250,0,1,0
3,10888400,0,0,1
4,10808240,0,0,1
5,10876543,1,0,0
6,10834282,1,0,0
7,10847978,1,0,0
8,10848108,0,0,1
9,10848095,0,0,1


# Pre-processing 

In [5]:
model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")

In [6]:
passage_train = train_df['translated_product'].to_numpy()
passage_embeddings_train = model.encode(passage_train)

passage_test = test_df['translated_product'].to_numpy()
passage_embeddings_test = model.encode(passage_test)

In [62]:
onehot = ['brand','category']
preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown='ignore'), onehot),
    # ("num", StandardScaler(), ['price'])
])
X = preprocessor.fit_transform(train_df).toarray()
num_categories = sum(len(c) for c in preprocessor.named_transformers_['cat'].categories_)
print(f"Number of categories after one-hot encoding: {num_categories}")

Number of categories after one-hot encoding: 764


In [63]:
train_df_processed = pd.DataFrame(X, columns=preprocessor.get_feature_names_out())
train_df_processed = pd.merge(train_df, train_df_processed, left_index=True, right_index=True)
cat_features_train = train_df_processed.values[:,-num_categories:]
X_final_train = np.hstack((passage_embeddings_train, cat_features_train))

test_df_processed = pd.DataFrame(preprocessor.transform(test_df).toarray(), columns=preprocessor.get_feature_names_out())
test_df_processed = pd.merge(test_df, test_df_processed, left_index=True, right_index=True)
cat_features_test = test_df_processed.values[:,-num_categories:]
X_final_test = np.hstack((passage_embeddings_test, cat_features_test))

# Train/test split

In [65]:
# train, val = train_test_split(X_final_train, test_size=0.2, random_state=42)
threshold = int(0.8 * X_final_train.shape[0])
val   = X_final_train[threshold:]
train = X_final_train[:threshold]
test  = X_final_test

# Create DataFrames with embeddings + category features
embedding_cols = [f'emb_{i}' for i in range(passage_embeddings_train.shape[1])]
cat_cols = preprocessor.get_feature_names_out().tolist()
all_cols = embedding_cols + cat_cols

train_df_split = pd.DataFrame(train,columns=all_cols)
val_df_split   = pd.DataFrame(val,  columns=all_cols)
test_df_split  = pd.DataFrame(test, columns=all_cols)

# Merge with original metadata (data_id, product_name, etc.)
train_indices = train_df_processed.index[~train_df_processed.index.isin(val_df_split.index)].tolist()
val_indices   = train_df_processed.index[ train_df_processed.index.isin(val_df_split.index)].tolist()

train = pd.concat([train_df_processed.iloc[train_indices].reset_index(drop=True), train_df_split], axis=1)
val   = pd.concat([train_df_processed.iloc[val_indices  ].reset_index(drop=True), val_df_split]  , axis=1)
test  = pd.concat([test_df_processed.reset_index(drop=True), test_df_split], axis=1)

In [66]:
train.shape, val.shape, test.shape

((1888, 2567), (473, 2567), (153, 2568))

In [71]:
list(train.columns)

['data_id',
 'price',
 'brand',
 'category',
 'product_name',
 'units',
 'quantity',
 'unit_type',
 'store_name',
 'start_date',
 'end_date',
 'public_urls',
 'translated_product',
 'tinder_bio',
 'image_url',
 'cat__brand_0 Kalorier',
 'cat__brand_1-Enkelt',
 'cat__brand_12 YO',
 'cat__brand_16 serie',
 'cat__brand_1883',
 'cat__brand_A+',
 'cat__brand_ASP',
 'cat__brand_Actimel',
 'cat__brand_Aftenstunder',
 'cat__brand_Agnes',
 'cat__brand_AhornBryg',
 'cat__brand_Ajax',
 'cat__brand_Ama',
 'cat__brand_Amarena Fabbri',
 'cat__brand_American Pale Ale',
 'cat__brand_Amo',
 'cat__brand_Amper Energy Drink',
 'cat__brand_Andoni',
 'cat__brand_Anthon Berg',
 'cat__brand_Antioxidant',
 'cat__brand_Antioxidant Fersken',
 'cat__brand_Aperol',
 'cat__brand_Apple',
 'cat__brand_Arla Protein',
 'cat__brand_Aruna',
 'cat__brand_Athena',
 'cat__brand_BKI',
 'cat__brand_Bacon',
 'cat__brand_Bacon Peber',
 'cat__brand_Bagernes Bedste',
 'cat__brand_Bakersfield',
 'cat__brand_Bakkedal',
 'cat__brand

In [ ]:
ppdf = pd.read_csv("../data/csv/processed/products_2025-12-20.csv")

In [72]:
list(test.columns)

['data_id',
 'price',
 'brand',
 'category',
 'product_name',
 'units',
 'quantity',
 'unit_type',
 'store_name',
 'image_url',
 'start_date',
 'end_date',
 'public_urls',
 'translated_product',
 'translated_category',
 'tinder_bio',
 'cat__brand_0 Kalorier',
 'cat__brand_1-Enkelt',
 'cat__brand_12 YO',
 'cat__brand_16 serie',
 'cat__brand_1883',
 'cat__brand_A+',
 'cat__brand_ASP',
 'cat__brand_Actimel',
 'cat__brand_Aftenstunder',
 'cat__brand_Agnes',
 'cat__brand_AhornBryg',
 'cat__brand_Ajax',
 'cat__brand_Ama',
 'cat__brand_Amarena Fabbri',
 'cat__brand_American Pale Ale',
 'cat__brand_Amo',
 'cat__brand_Amper Energy Drink',
 'cat__brand_Andoni',
 'cat__brand_Anthon Berg',
 'cat__brand_Antioxidant',
 'cat__brand_Antioxidant Fersken',
 'cat__brand_Aperol',
 'cat__brand_Apple',
 'cat__brand_Arla Protein',
 'cat__brand_Aruna',
 'cat__brand_Athena',
 'cat__brand_BKI',
 'cat__brand_Bacon',
 'cat__brand_Bacon Peber',
 'cat__brand_Bagernes Bedste',
 'cat__brand_Bakersfield',
 'cat__brand

# Inference

In [46]:
train = pd.merge(train, labels, on='data_id', how='inner')
val   = pd.merge(val,   labels, on='data_id', how='inner')

### FAISS

In [30]:
# Build FAISS index for cosine similarity over translated product embeddings
train_vecs = np.ascontiguousarray(train_df_split.astype('float32'))
test_vecs  = np.ascontiguousarray(test_df_split.astype('float32'))

# Normalize for cosine similarity (dot-product index)
faiss.normalize_L2(train_vecs)
faiss.normalize_L2(test_vecs)

index = faiss.IndexFlatIP(train_vecs.shape[1])
index.add(train_vecs)
print(f"Indexed {index.ntotal} product vectors")

# Example lookup: top-5 nearest neighbors for first test item
D, I = index.search(test_vecs[100].reshape(1,-1), 5)
print("Top-5 neighbors (indices):", I[0])
print("Similarity scores:", D[0])

Indexed 1888 product vectors
Top-5 neighbors (indices): [1072 1517 1579 1003 1576]
Similarity scores: [1.         0.6666666  0.58235776 0.5730605  0.56953543]


In [31]:
test_df.iloc[100].values

array([np.int64(10888481), np.float64(20.0), 'Klovborg', 'Skiveost',
       'Skiveost Danbo Mellemlagret', np.int64(1), np.int64(0), 'pk.',
       'Netto',
       'https://static.tilbudsugen.dk/1st-retail/2025/51/64237/Netto522025-1_13_70x1051_2823x2736_zoom.jpg',
       '2025-12-20', '2025-12-23',
       'https://res.cloudinary.com/dfqzmnlga/image/upload/v1766229456/2025-12-20/10888481.jpg',
       'Skiveost Danbo Intermediate stock', 'Skiver cheese',
       'Skiveost Danbo Intermediate stock: The ultimate multitasker who can handle any task with ease and a dash of flair! 🌟'],
      dtype=object)

In [39]:
train_df.iloc[1576].values

array([np.int64(10847954), np.float64(25.0), 'Løgismose', 'Skiveost',
       'Skiveost Aged Havarti', np.int64(1), np.int64(0), 'pk.', 'Netto',
       '2025-11-29', '2025-12-05',
       'https://res.cloudinary.com/dfqzmnlga/image/upload/v1764415614/2025-11-29/10847954.jpg',
       'Skiveost Aged Havarti',
       'Savory, slightly cheesy, and always ready to melt into a perfect bite 🧀',
       'https://static.tilbudsugen.dk/1st-retail/2025/49/64080/Netto492025-1_11_1692x5187_3332x6870_zoom.jpg'],
      dtype=object)

### KNN

In [49]:
train

,data_id,price,brand,category,product_name,units,quantity,unit_type,store_name,start_date,...,cat__category_Vitaminer,cat__category_Vodka,"cat__category_Våben, udklædning, rollespil",cat__category_Whisky,cat__category_Øl,cat__category_Øvrige mejeriprodukter,cat__category_Øvrige vinlande,is_liked,is_superliked,is_passed
0,10820949,16.00,Juleleverpostej,Postej og Paté,Leverpostej,1,0,pk.,Netto,2025-11-15,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,1
1,10820865,10.00,Gul bordpak,"Pålæg, skiveskåret",Sønderjysk Spegepølse,1,0,pk.,Netto,2025-11-15,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0,0
2,10834282,10.00,Faxe Kondi,Sodavand,Sportssodavand,1,1,fl.,Netto,2025-11-22,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0,0
3,10834229,25.00,Magnum,Multipak,Classic ispinde,1,3,pk.,Netto,2025-11-22,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,1
4,10833997,10.00,Arla Protein,Mellemmåltider,Proteindrik Chocolate,1,0,fl.,Netto,2025-11-22,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0,0
5,10834191,12.00,Sønderjyske Fristelser,Pålægssalat,Skagensalat,1,0,bg.,Netto,2025-11-22,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0,0
6,10834173,12.00,Sønderjyske Fristelser,Pålægssalat,Kylling & Baconsalat,1,0,bg.,Netto,2025-11-22,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0,0
7,10833978,10.00,Salami Hapser,Pålægspølser,Salami Snacks,1,0,pk.,Netto,2025-11-22,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,1
8,10834083,30.00,NaN,Skriveartikler,Penalhus,1,0,stk.,Netto,2025-11-22,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,1
9,10848109,10.00,ElleBryg Jul,Alkoholfrie Øl,Øl - Alkoholfri,1,0,fl.,Netto,2025-11-29,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,1
